# FoodCorp Churn Prediction Using Temporal Behavioural Features

## Portfolio notebook

This notebook documents a churn-prediction workflow for FoodCorp. The objective is to identify customers who are still active at a weekly reference date but are at risk of becoming inactive.

**Portfolio data note:** raw transaction data is not included in this public version. To run this notebook, the following source tables need to exist in a Spark/Databricks environment:

- `customers`
- `products`
- `receipts`
- `receipt_lines`

The workflow is structured to avoid temporal leakage: features are calculated only from behaviour observed on or before each weekly reference date, while the churn label is calculated from future behaviour after that date.

## 1. Transaction and product-quality controls

The source data contains some transaction rows that should not be treated as normal customer behaviour, including negative quantities/values and extreme corrupted values. These are excluded before feature engineering so that the churn model is not driven by operational or data-entry artefacts.

In [ ]:
CREATE OR REPLACE TEMP VIEW receipt_lines_clean AS
SELECT *
FROM receipt_lines
WHERE qty > 0
  AND value >= 0
  AND value < 10000;

In [ ]:
CREATE OR REPLACE TEMP VIEW clean_receipts AS
SELECT DISTINCT r.*
FROM receipts r
JOIN receipt_lines_clean rl
  ON r.receipt_id = rl.receipt_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW products_clean AS
SELECT *
FROM products
WHERE department_code >= 0
  AND category_code >= 0
  AND sub_category_code >= 0;

## 2. Churn definition

Churn is defined as **no purchase activity for more than 35 days**. Candidate thresholds were reviewed before choosing this value. A shorter threshold would identify risk earlier but may over-label natural shopping gaps as churn; a longer threshold would delay intervention.

In [ ]:
CREATE OR REPLACE TEMP VIEW customer_purchase_days AS
SELECT DISTINCT
  customer_id,
  purchased_at AS purchase_date
FROM clean_receipts;

In [ ]:
CREATE OR REPLACE TEMP VIEW customer_purchase_gaps AS
SELECT customer_id ,
       purchase_date,  
       lag (purchase_date) over(partition by customer_id order by purchase_date) as previous_purchase,
       datediff( purchase_date , lag (purchase_date) over(partition by customer_id order by purchase_date) ) as gap
FROM customer_purchase_days

In [ ]:
CREATE OR REPLACE TEMP VIEW customer_median_gap AS
SELECT
  customer_id,
  percentile_approx(gap, 0.5) AS median_gap_days,
  count(gap) AS n_gaps
FROM customer_purchase_gaps
WHERE gap IS NOT NULL
GROUP BY customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW calendar_dates AS
SELECT explode(
  sequence(
    (SELECT MIN(purchased_at) FROM clean_receipts),
    (SELECT MAX(purchased_at) FROM clean_receipts),interval 1 day)) 
AS ref_date;

In [ ]:
CREATE OR REPLACE TEMP VIEW beta_candidates AS
SELECT explode(sequence(7, 84, 7)) AS beta_days;

In [ ]:
CREATE OR REPLACE TEMP VIEW valid_ref_dates AS
SELECT ref_date
FROM calendar_dates
WHERE ref_date <= DATE_SUB(
  (SELECT MAX(purchased_at) FROM clean_receipts),
  (SELECT MAX(beta_days) FROM beta_candidates)
);

In [ ]:
CREATE OR REPLACE TEMP VIEW customer_last_purchase_before_ref AS
SELECT c.customer_id, 
       v.ref_date , MAX(c.purchase_date) AS last_purchase_date
FROM customer_purchase_days c 
JOIN valid_ref_dates v
ON c.purchase_date <= v.ref_date
GROUP BY v.ref_date, c.customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW customer_inactivity_till_ref AS
SELECT
  customer_id,
  ref_date,
  last_purchase_date,
  DATEDIFF(ref_date, last_purchase_date) AS inactivity_days
FROM customer_last_purchase_before_ref;

In [ ]:
CREATE OR REPLACE TEMP VIEW customer_next_purchase_after_ref AS
SELECT
  c.customer_id,
  v.ref_date,
  MIN(c.purchase_date) AS next_purchase_date
FROM valid_ref_dates v
JOIN customer_purchase_days c
  ON c.purchase_date > v.ref_date
GROUP BY v.ref_date, c.customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW customer_purchase_interval AS
SELECT
  customer_id,
  purchase_date AS interval_start_date,
  LEAD(purchase_date) OVER (
    PARTITION BY customer_id
    ORDER BY purchase_date
  ) AS next_purchase_date,
  DATEDIFF(
    LEAD(purchase_date) OVER (
      PARTITION BY customer_id
      ORDER BY purchase_date
    ),
    purchase_date
  ) AS gap
FROM customer_purchase_days;

In [ ]:
CREATE OR REPLACE TEMP VIEW interval_beta_status AS
SELECT
  i.customer_id,
  i.interval_start_date AS ref_date,
  b.beta_days,
  i.next_purchase_date,
  i.gap,
  CASE
    WHEN i.next_purchase_date IS NULL
         OR i.gap > b.beta_days
    THEN 1
    ELSE 0
  END AS churn_event_on_ref_date
FROM customer_purchase_interval i
JOIN valid_ref_dates v
  ON i.interval_start_date = v.ref_date
CROSS JOIN beta_candidates b;

In [ ]:
CREATE OR REPLACE TEMP VIEW interval_active_ranges AS
SELECT
  customer_id,
  beta_days,
  ref_date AS active_start_date,
  CASE
    WHEN churn_event_on_ref_date = 1 THEN ref_date
    ELSE DATE_SUB(next_purchase_date, 1)
  END AS active_end_date,
  churn_event_on_ref_date
FROM interval_beta_status;

In [ ]:
CREATE OR REPLACE TEMP VIEW active_customers_by_ref AS
SELECT
  i.customer_id,
  v.ref_date,
  i.beta_days
FROM interval_active_ranges i
JOIN valid_ref_dates v
  ON v.ref_date BETWEEN i.active_start_date AND i.active_end_date;

In [ ]:
CREATE OR REPLACE TEMP VIEW churn_events_by_ref AS
SELECT
  ref_date,
  beta_days,
  COUNT(DISTINCT customer_id) AS churn_events
FROM interval_beta_status
WHERE churn_event_on_ref_date = 1
GROUP BY ref_date, beta_days;

In [ ]:
CREATE OR REPLACE TEMP VIEW ref_date_beta_summary_v2 AS
SELECT
  a.ref_date,
  a.beta_days,
  COUNT(DISTINCT a.customer_id) AS active_customers,
  COALESCE(c.churn_events, 0) AS churn_events,
  ROUND(
    100.0 * COALESCE(c.churn_events, 0) / COUNT(DISTINCT a.customer_id),
    2
  ) AS churn_pct_among_active
FROM active_customers_by_ref a
LEFT JOIN churn_events_by_ref c
  ON a.ref_date = c.ref_date
 AND a.beta_days = c.beta_days
GROUP BY
  a.ref_date,
  a.beta_days,
  c.churn_events;

In [ ]:
SELECT
  beta_days,
  ROUND(AVG(active_customers), 2) AS avg_active_customers,
  ROUND(AVG(churn_events), 2) AS avg_churn_events,
  ROUND(AVG(churn_pct_among_active), 2) AS avg_churn_pct_among_active
FROM ref_date_beta_summary_v2
WHERE beta_days IN (21, 28, 35, 42)
GROUP BY beta_days
ORDER BY beta_days;

In [ ]:
CREATE OR REPLACE TEMP VIEW chosen_beta AS
SELECT 35 AS beta_days;

## 3. Weekly modelling base

A weekly modelling table is created so that each row represents one active customer at one weekly reference date. The model is only asked to predict customers while they are still active and therefore still actionable.

In [ ]:
CREATE OR REPLACE TEMP VIEW weekly_ref_dates AS
SELECT ref_date
FROM valid_ref_dates
WHERE dayofweek(ref_date) = 1;

In [ ]:
CREATE OR REPLACE TEMP VIEW weekly_customer_base AS
SELECT
  cir.ref_date,
  cir.customer_id,
  cb.beta_days,
  cir.last_purchase_date,
  cir.inactivity_days,
  cnp.next_purchase_date,
  DATEDIFF(cnp.next_purchase_date, cir.ref_date) AS days_until_next_purchase,
  CASE
    WHEN cir.inactivity_days <= cb.beta_days THEN 1
    ELSE 0
  END AS is_active_at_ref,
  CASE
    WHEN cir.inactivity_days <= cb.beta_days
         AND (
           cnp.next_purchase_date IS NULL
           OR DATEDIFF(cnp.next_purchase_date, cir.ref_date) > cb.beta_days
         )
    THEN 1
    ELSE 0
  END AS churn_label
FROM customer_inactivity_till_ref cir
JOIN weekly_ref_dates w
  ON cir.ref_date = w.ref_date
CROSS JOIN chosen_beta cb
LEFT JOIN customer_next_purchase_after_ref cnp
  ON cir.ref_date = cnp.ref_date
 AND cir.customer_id = cnp.customer_id
WHERE cir.inactivity_days <= cb.beta_days;

In [ ]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customer_id) AS distinct_customers,
  COUNT(DISTINCT ref_date) AS distinct_ref_dates,
  SUM(churn_label) AS positive_labels,
  ROUND(100.0 * SUM(churn_label) / COUNT(*), 2) AS positive_label_pct
FROM weekly_customer_base;

In [ ]:
SELECT
  churn_label,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT customer_id) AS n_customers,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM weekly_customer_base
GROUP BY churn_label
ORDER BY churn_label;

## 4. SQL feature engineering

The feature set captures four behavioural dimensions:

1. **Recency** — days since the customer's most recent purchase.
2. **Frequency** — distinct purchase days over 7, 35 and 70 days.
3. **Monetary value and quantity** — spend and quantity over the same windows.
4. **Category breadth and trend** — how broad and stable the customer's shopping behaviour is over time.

In [ ]:
CREATE OR REPLACE TEMP VIEW feature_recency AS
SELECT
  ref_date,
  customer_id,
  inactivity_days AS recency_days
FROM weekly_customer_base;

In [ ]:
CREATE OR REPLACE TEMP VIEW feature_frequency AS
SELECT
  b.ref_date,
  b.customer_id,
  SUM(CASE WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 6)  AND b.ref_date THEN 1 ELSE 0 END) AS purchase_days_7d,
  SUM(CASE WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 34) AND b.ref_date THEN 1 ELSE 0 END) AS purchase_days_35d,
  SUM(CASE WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 69) AND b.ref_date THEN 1 ELSE 0 END) AS purchase_days_70d
FROM weekly_customer_base b
LEFT JOIN customer_purchase_days cpd
  ON b.customer_id = cpd.customer_id
 AND cpd.purchase_date <= b.ref_date
GROUP BY
  b.ref_date,
  b.customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW feature_monetary AS
SELECT
  b.ref_date,
  b.customer_id,
  SUM(CASE WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 6)  AND b.ref_date THEN rl.value ELSE 0 END) AS spend_7d,
  SUM(CASE WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 34) AND b.ref_date THEN rl.value ELSE 0 END) AS spend_35d,
  SUM(CASE WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 69) AND b.ref_date THEN rl.value ELSE 0 END) AS spend_70d
FROM weekly_customer_base b
LEFT JOIN clean_receipts r
  ON b.customer_id = r.customer_id
 AND r.purchased_at <= b.ref_date
LEFT JOIN receipt_lines_clean rl
  ON r.receipt_id = rl.receipt_id
GROUP BY
  b.ref_date,
  b.customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW feature_quantity AS
SELECT
  b.ref_date,
  b.customer_id,
  SUM(CASE WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 6)  AND b.ref_date THEN rl.qty ELSE 0 END) AS qty_7d,
  SUM(CASE WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 34) AND b.ref_date THEN rl.qty ELSE 0 END) AS qty_35d,
  SUM(CASE WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 69) AND b.ref_date THEN rl.qty ELSE 0 END) AS qty_70d
FROM weekly_customer_base b
LEFT JOIN clean_receipts r
  ON b.customer_id = r.customer_id
 AND r.purchased_at <= b.ref_date
LEFT JOIN receipt_lines_clean rl
  ON r.receipt_id = rl.receipt_id
GROUP BY
  b.ref_date,
  b.customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW feature_diversity AS
SELECT
  b.ref_date,
  b.customer_id,
  COUNT(DISTINCT CASE
    WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 34) AND b.ref_date
    THEN p.category_details
  END) AS category_count_35d,
  COUNT(DISTINCT CASE
    WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 69) AND b.ref_date
    THEN p.category_details
  END) AS category_count_70d
FROM weekly_customer_base b
LEFT JOIN clean_receipts r
  ON b.customer_id = r.customer_id
 AND r.purchased_at <= b.ref_date
LEFT JOIN receipt_lines_clean rl
  ON r.receipt_id = rl.receipt_id
LEFT JOIN products_clean p
  ON rl.product_code = p.product_code
GROUP BY
  b.ref_date,
  b.customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW feature_frequency_trend AS
SELECT
  b.ref_date,
  b.customer_id,

  SUM(CASE
        WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 34) AND b.ref_date
        THEN 1 ELSE 0
      END) AS purchase_days_recent_35d,
  SUM(CASE
        WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 69) AND DATE_SUB(b.ref_date, 35)
        THEN 1 ELSE 0
      END) AS purchase_days_prev_35d,
  SUM(CASE
        WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 34) AND b.ref_date
        THEN 1 ELSE 0
      END)
  -
  SUM(CASE
        WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 69) AND DATE_SUB(b.ref_date, 35)
        THEN 1 ELSE 0
      END) AS purchase_days_change_35d,

  SUM(CASE
        WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 69) AND b.ref_date
        THEN 1 ELSE 0
      END) AS purchase_days_recent_70d,
  SUM(CASE
        WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 139) AND DATE_SUB(b.ref_date, 70)
        THEN 1 ELSE 0
      END) AS purchase_days_prev_70d,
  SUM(CASE
        WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 69) AND b.ref_date
        THEN 1 ELSE 0
      END)
  -
  SUM(CASE
        WHEN cpd.purchase_date BETWEEN DATE_SUB(b.ref_date, 139) AND DATE_SUB(b.ref_date, 70)
        THEN 1 ELSE 0
      END) AS purchase_days_change_70d

FROM weekly_customer_base b
LEFT JOIN customer_purchase_days cpd
  ON b.customer_id = cpd.customer_id
 AND cpd.purchase_date <= b.ref_date
GROUP BY
  b.ref_date,
  b.customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW feature_monetary_trend AS
SELECT
  b.ref_date,
  b.customer_id,

  SUM(CASE
        WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 34) AND b.ref_date
        THEN rl.value ELSE 0
      END) AS spend_recent_35d,
  SUM(CASE
        WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 69) AND DATE_SUB(b.ref_date, 35)
        THEN rl.value ELSE 0
      END) AS spend_prev_35d,
  SUM(CASE
        WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 34) AND b.ref_date
        THEN rl.value ELSE 0
      END)
  -
  SUM(CASE
        WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 69) AND DATE_SUB(b.ref_date, 35)
        THEN rl.value ELSE 0
      END) AS spend_change_35d,

  SUM(CASE
        WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 69) AND b.ref_date
        THEN rl.value ELSE 0
      END) AS spend_recent_70d,
  SUM(CASE
        WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 139) AND DATE_SUB(b.ref_date, 70)
        THEN rl.value ELSE 0
      END) AS spend_prev_70d,
  SUM(CASE
        WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 69) AND b.ref_date
        THEN rl.value ELSE 0
      END)
  -
  SUM(CASE
        WHEN r.purchased_at BETWEEN DATE_SUB(b.ref_date, 139) AND DATE_SUB(b.ref_date, 70)
        THEN rl.value ELSE 0
      END) AS spend_change_70d

FROM weekly_customer_base b
LEFT JOIN clean_receipts r
  ON b.customer_id = r.customer_id
 AND r.purchased_at <= b.ref_date
LEFT JOIN receipt_lines_clean rl
  ON r.receipt_id = rl.receipt_id
GROUP BY
  b.ref_date,
  b.customer_id;

In [ ]:
CREATE OR REPLACE TEMP VIEW final_feature_table AS
SELECT
  b.ref_date,
  b.customer_id,
  b.beta_days,
  b.churn_label,

  r.recency_days,

  f.purchase_days_7d,
  f.purchase_days_35d,
  f.purchase_days_70d,

  m.spend_7d,
  m.spend_35d,
  m.spend_70d,

  q.qty_7d,
  q.qty_35d,
  q.qty_70d,

  d.category_count_35d,
  d.category_count_70d,

  ft.purchase_days_recent_35d,
  ft.purchase_days_prev_35d,
  ft.purchase_days_change_35d,
  ft.purchase_days_recent_70d,
  ft.purchase_days_prev_70d,
  ft.purchase_days_change_70d,

  mt.spend_recent_35d,
  mt.spend_prev_35d,
  mt.spend_change_35d,
  mt.spend_recent_70d,
  mt.spend_prev_70d,
  mt.spend_change_70d

FROM weekly_customer_base b
LEFT JOIN feature_recency r
  ON b.ref_date = r.ref_date
 AND b.customer_id = r.customer_id
LEFT JOIN feature_frequency f
  ON b.ref_date = f.ref_date
 AND b.customer_id = f.customer_id
LEFT JOIN feature_monetary m
  ON b.ref_date = m.ref_date
 AND b.customer_id = m.customer_id
LEFT JOIN feature_quantity q
  ON b.ref_date = q.ref_date
 AND b.customer_id = q.customer_id
LEFT JOIN feature_diversity d
  ON b.ref_date = d.ref_date
 AND b.customer_id = d.customer_id
LEFT JOIN feature_frequency_trend ft
  ON b.ref_date = ft.ref_date
 AND b.customer_id = ft.customer_id
LEFT JOIN feature_monetary_trend mt
  ON b.ref_date = mt.ref_date
 AND b.customer_id = mt.customer_id;

In [ ]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customer_id) AS distinct_customers,
  COUNT(DISTINCT ref_date) AS distinct_ref_dates,
  SUM(churn_label) AS positive_labels,
  ROUND(100.0 * SUM(churn_label) / COUNT(*), 2) AS positive_label_pct
FROM final_feature_table;

In [ ]:
SELECT
  MIN(recency_days) AS min_recency_days,
  MAX(recency_days) AS max_recency_days,
  AVG(recency_days) AS avg_recency_days,

  MIN(purchase_days_35d) AS min_purchase_days_35d,
  MAX(purchase_days_35d) AS max_purchase_days_35d,
  AVG(purchase_days_35d) AS avg_purchase_days_35d,

  MIN(spend_35d) AS min_spend_35d,
  MAX(spend_35d) AS max_spend_35d,
  AVG(spend_35d) AS avg_spend_35d,

  MIN(qty_35d) AS min_qty_35d,
  MAX(qty_35d) AS max_qty_35d,
  AVG(qty_35d) AS avg_qty_35d,

  MIN(category_count_35d) AS min_category_count_35d,
  MAX(category_count_35d) AS max_category_count_35d,
  AVG(category_count_35d) AS avg_category_count_35d
FROM final_feature_table;

## 5. Modelling setup in Python

The final modelling table is moved from Spark into pandas. Non-predictive identifiers are excluded from the model input, while `churn_label` is retained as the target variable.

In [ ]:
%python
import pandas as pd
import numpy as np

df_model = spark.table("final_feature_table").toPandas()
df_model["ref_date"] = pd.to_datetime(df_model["ref_date"])

print("Modelling table shape:", df_model.shape)
print("Churn-label distribution:")
print(df_model["churn_label"].value_counts(normalize=True).sort_index())

In [ ]:
%python
sorted_dates = sorted(df_model["ref_date"].unique())

# Final 8 weekly dates are held out as the untouched final test period.
train_valid_dates = sorted_dates[:-8]
test_dates = sorted_dates[-8:]

train_valid_df = df_model[df_model["ref_date"].isin(train_valid_dates)].copy()
test_df = df_model[df_model["ref_date"].isin(test_dates)].copy()

print("Total weekly reference dates:", len(sorted_dates))
print("Development weeks:", len(train_valid_dates))
print("Final holdout weeks:", len(test_dates))
print("Development rows:", len(train_valid_df))
print("Holdout rows:", len(test_df))

In [ ]:
%python
# Three expanding out-of-time validation folds are used within the development period.
train_valid_sorted_dates = sorted(train_valid_df["ref_date"].unique())

folds = [
    {
        "fold_name": "Fold 1",
        "train_dates": train_valid_sorted_dates[:43],
        "val_dates": train_valid_sorted_dates[43:51]
    },
    {
        "fold_name": "Fold 2",
        "train_dates": train_valid_sorted_dates[:51],
        "val_dates": train_valid_sorted_dates[51:59]
    },
    {
        "fold_name": "Fold 3",
        "train_dates": train_valid_sorted_dates[:59],
        "val_dates": train_valid_sorted_dates[59:67]
    }
]

for fold in folds:
    print(
        fold["fold_name"],
        "| train weeks:", len(fold["train_dates"]),
        "| validation weeks:", len(fold["val_dates"])
    )

In [ ]:
%python
non_feature_cols = ["ref_date", "customer_id", "beta_days", "churn_label"]
target_col = "churn_label"

feature_cols = [c for c in df_model.columns if c not in non_feature_cols]

print("Number of candidate feature columns:", len(feature_cols))
print(feature_cols)

## 6. Model comparison

Logistic Regression and Random Forest are compared using the same temporal folds. Logistic Regression is scaled inside a pipeline so that scaling is learned only from each training fold and then applied to later validation data.

In [ ]:
%python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_predictions(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

def evaluate_logistic_fold(train_df, val_df, feature_cols, target_col):
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()
    X_val = val_df[feature_cols].copy()
    y_val = val_df[target_col].copy()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    return evaluate_predictions(y_val, y_pred)

def evaluate_rf_fold(train_df, val_df, feature_cols, target_col):
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()
    X_val = val_df[feature_cols].copy()
    y_val = val_df[target_col].copy()

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    return evaluate_predictions(y_val, y_pred)

In [ ]:
%python
def run_temporal_evaluation(model_name, evaluator):
    rows = []

    for fold in folds:
        train_df_fold = train_valid_df[train_valid_df["ref_date"].isin(fold["train_dates"])].copy()
        val_df_fold = train_valid_df[train_valid_df["ref_date"].isin(fold["val_dates"])].copy()

        metrics = evaluator(train_df_fold, val_df_fold, feature_cols, target_col)

        rows.append({
            "model": model_name,
            "fold": fold["fold_name"],
            "train_rows": len(train_df_fold),
            "validation_rows": len(val_df_fold),
            **{k: round(v, 4) for k, v in metrics.items()}
        })

    return pd.DataFrame(rows)

logistic_results_df = run_temporal_evaluation("Logistic Regression", evaluate_logistic_fold)
rf_results_df = run_temporal_evaluation("Random Forest", evaluate_rf_fold)

model_comparison_df = pd.concat([logistic_results_df, rf_results_df], ignore_index=True)
model_comparison_df

In [ ]:
%python
model_comparison_summary = (
    model_comparison_df
    .groupby("model")[["accuracy", "precision", "recall", "f1"]]
    .mean()
    .round(4)
    .reset_index()
)

model_comparison_summary

## 7. Logistic Regression tuning

Logistic Regression is selected because it provides strong temporal validation performance and direct interpretation through coefficients. Manual tuning tests regularisation strength (`C`) and class weighting using only the development folds.

In [ ]:
%python
param_grid = [
    {"C": 0.1, "class_weight": None},
    {"C": 1.0, "class_weight": None},
    {"C": 3.0, "class_weight": None},
    {"C": 10.0, "class_weight": None},
    {"C": 0.1, "class_weight": "balanced"},
    {"C": 1.0, "class_weight": "balanced"},
    {"C": 3.0, "class_weight": "balanced"},
    {"C": 10.0, "class_weight": "balanced"},
]

tuning_results = []

for params in param_grid:
    fold_metrics = []

    for fold in folds:
        train_df_fold = train_valid_df[train_valid_df["ref_date"].isin(fold["train_dates"])].copy()
        val_df_fold = train_valid_df[train_valid_df["ref_date"].isin(fold["val_dates"])].copy()

        X_train = train_df_fold[feature_cols].copy()
        y_train = train_df_fold[target_col].copy()
        X_val = val_df_fold[feature_cols].copy()
        y_val = val_df_fold[target_col].copy()

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=params["C"],
                class_weight=params["class_weight"],
                max_iter=1000,
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        fold_metrics.append(evaluate_predictions(y_val, y_pred))

    fold_metrics_df = pd.DataFrame(fold_metrics)

    tuning_results.append({
        "C": params["C"],
        "class_weight": params["class_weight"],
        "avg_precision": round(fold_metrics_df["precision"].mean(), 4),
        "avg_recall": round(fold_metrics_df["recall"].mean(), 4),
        "avg_f1": round(fold_metrics_df["f1"].mean(), 4)
    })

tuning_results_df = (
    pd.DataFrame(tuning_results)
    .sort_values("avg_f1", ascending=False)
    .reset_index(drop=True)
)

tuning_results_df

In [ ]:
%python
threshold_grid = np.arange(0.30, 0.71, 0.05)
threshold_results = []

for threshold in threshold_grid:
    fold_metrics = []

    for fold in folds:
        train_df_fold = train_valid_df[train_valid_df["ref_date"].isin(fold["train_dates"])].copy()
        val_df_fold = train_valid_df[train_valid_df["ref_date"].isin(fold["val_dates"])].copy()

        X_train = train_df_fold[feature_cols].copy()
        y_train = train_df_fold[target_col].copy()
        X_val = val_df_fold[feature_cols].copy()
        y_val = val_df_fold[target_col].copy()

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=3.0,
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)
        y_val_prob = model.predict_proba(X_val)[:, 1]
        y_val_pred = (y_val_prob >= threshold).astype(int)

        fold_metrics.append(evaluate_predictions(y_val, y_val_pred))

    fold_metrics_df = pd.DataFrame(fold_metrics)

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "avg_precision": round(fold_metrics_df["precision"].mean(), 4),
        "avg_recall": round(fold_metrics_df["recall"].mean(), 4),
        "avg_f1": round(fold_metrics_df["f1"].mean(), 4)
    })

threshold_results_df = (
    pd.DataFrame(threshold_results)
    .sort_values("avg_f1", ascending=False)
    .reset_index(drop=True)
)

threshold_results_df

## 8. Final model and holdout evaluation

Four exact duplicate window features are removed before fitting the final model. The final tuned model is then trained on the full development period and evaluated once on the untouched final 8-week holdout period.

In [ ]:
%python
duplicate_feature_cols = [
    "purchase_days_recent_35d",
    "purchase_days_recent_70d",
    "spend_recent_35d",
    "spend_recent_70d"
]

feature_cols_clean = [c for c in feature_cols if c not in duplicate_feature_cols]

print("Original feature count:", len(feature_cols))
print("Clean feature count:", len(feature_cols_clean))
print("Dropped duplicate features:", duplicate_feature_cols)

In [ ]:
%python
X_train_final = train_valid_df[feature_cols_clean].copy()
y_train_final = train_valid_df[target_col].copy()

X_test_final = test_df[feature_cols_clean].copy()
y_test_final = test_df[target_col].copy()

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=3.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

final_model.fit(X_train_final, y_train_final)
y_test_pred = final_model.predict(X_test_final)

final_metrics = {
    "accuracy": round(accuracy_score(y_test_final, y_test_pred), 4),
    "precision": round(precision_score(y_test_final, y_test_pred), 4),
    "recall": round(recall_score(y_test_final, y_test_pred), 4),
    "f1": round(f1_score(y_test_final, y_test_pred), 4)
}

print(final_metrics)
print(confusion_matrix(y_test_final, y_test_pred))

**Final holdout result reported in the portfolio case study**

| Metric | Value |
|---|---:|
| Accuracy | 0.7824 |
| Precision | 0.6667 |
| Recall | 0.8681 |
| F1 score | 0.7542 |
| Confusion matrix | [[8930, 3324], [1010, 6649]] |

## 9. Model interpretation

The final model is interpreted using two complementary methods:

- **Standardised Logistic Regression coefficients** to show direction and strength of association.
- **Permutation importance** to show how much each feature contributes to holdout F1 performance.

In [ ]:
%python
lr_model = final_model.named_steps["model"]

coef_df = pd.DataFrame({
    "feature": feature_cols_clean,
    "coefficient": lr_model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = (
    coef_df
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

coef_df.head(15)

In [ ]:
%python
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    estimator=final_model,
    X=X_test_final,
    y=y_test_final,
    scoring="f1",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_importance_df = pd.DataFrame({
    "feature": feature_cols_clean,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
})

perm_importance_df = (
    perm_importance_df
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

perm_importance_df.head(15)

In [ ]:
%python
insight_features = [
    "recency_days",
    "purchase_days_35d",
    "purchase_days_70d",
    "category_count_35d",
    "category_count_70d",
    "qty_35d",
    "qty_70d",
    "spend_35d",
    "spend_70d",
    "purchase_days_change_35d",
    "purchase_days_change_70d",
    "spend_change_35d",
    "spend_change_70d"
]

comparison_table = (
    test_df
    .groupby("churn_label")[insight_features]
    .mean()
    .T
    .reset_index()
    .rename(columns={
        "index": "feature",
        0: "non_churn_mean",
        1: "churn_mean"
    })
)

comparison_table["churn_minus_nonchurn"] = (
    comparison_table["churn_mean"] - comparison_table["non_churn_mean"]
)

comparison_table

## 10. Key conclusion

The strongest churn signals are declining shopping regularity and narrowing category engagement. Spend contributes information, but the model relies more heavily on whether customers continue to shop regularly and across multiple categories. This supports using the prediction system as an early-warning tool rather than a spend-only customer ranking.